In [0]:
storage_account_name = "hkhiarstore"
container_name = "hkhiar-datalake"
storage_account_key = dbutils.secrets.get(scope="azure_access_key",key="storage_key")
azure_key_conf = f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net"

base_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/"
silver_base_path = base_path + "Silver/"
gold_base_path = base_path + "Gold/"

try:
    print("Début du chargement des fichiers Parquet depuis la couche Silver...\n")
    
    df_silver = (spark.read
                    .format("delta")
                    .option(azure_key_conf, storage_account_key)
                    .load(f"{silver_base_path}/"))
    print("✔ Source 'Silver' chargée.")
    
    print("\n[SUCCÈS] a Dataframe de la couche Silver est prêt à être transformé !")
    
    # Petit affichage de contrôle pour vérifier qu'on a bien tout
    print("\n--- Structure du DataFrame Silver ---")
    df_silver.printSchema()
    df_silver.show(20)

    
except Exception as e:
    print("Détail de l'erreur :", e)

In [0]:
from pyspark.sql import functions as F

df_gold = (
    df_silver
    # Enrichissement temporel avant le groupBy
    .withColumn("year",  F.year(F.col("transaction_date")))
    .withColumn("month", F.month(F.col("transaction_date")))
 
    .groupBy(
        "product_id",
        "product_name",
        "category",
        "transaction_date",   # granularité jour
        "year",
        "month"
    )
    .agg(
        F.sum("quantity")                          .alias("total_quantity"),
        F.round(F.sum("total_amount"),       2)   .alias("total_revenue"),
        F.countDistinct("transaction_id")          .alias("nb_transactions"),
        F.round(
            F.sum("total_amount") / F.countDistinct("transaction_id"), 2
        )                                          .alias("avg_transaction_value")
    )
 
    # Tri lisible pour les contrôles visuels
    .orderBy("transaction_date", "category", "product_name")
)
 
print("Gold — lignes :", df_gold.count())
df_gold.printSchema()
display(df_gold.orderBy("product_id",ascending=True))

In [0]:
gold_path = gold_base_path 
 
try:
    (df_gold
        .write
        .format("delta")
        .option(azure_key_conf, storage_account_key)
        .mode("overwrite")
        .save(gold_path))
 
    print(f"✔ Gold écrite en Delta → {gold_path}")
    print("   (vérifier _delta_log/ + .snappy.parquet dans Azure Storage Browser)")
 
except Exception as e:
    print("[ERREUR] Échec de l'écriture Gold :", e)
    raise
